# Hover Logix detector: training round

Trains the next detector version from photos you approved in **Warehouse Detection Review**.

**Before you run**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. In Google Drive make a folder **HoverLogix** with two subfolders: **exports** and **models**.
3. Put every `hoverlogix_review_….zip` you exported into **HoverLogix/exports** (keep the old ones too: newer reviews of the same photo win).
4. First round only: put **hoverlogix_v1.pt** in **HoverLogix/models/v1/** so training starts from the current model.

Then **Runtime → Run all**. A round on ~1,000 photos takes about 40 to 90 minutes on a T4.
At the end, download **hoverlogix_model_vN.zip** from **HoverLogix/models/** and attach it in the chat with Claude,
who puts the new model into the review page and the apps.

In [ ]:
# 1. GPU, Drive, packages
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics onnx onnxslim

In [ ]:
# 2. Settings
import os, re, glob
DRIVE = '/content/drive/MyDrive/HoverLogix'
EXPORTS = f'{DRIVE}/exports'
MODELS = f'{DRIVE}/models'
IMGSZ = 960        # photos have small labels: keep 960
EPOCHS = 120
PATIENCE = 30      # stop early when validation stops improving
VAL_SHARE = 0.15
SEED = 7

os.makedirs(MODELS, exist_ok=True)
versions = sorted([int(m.group(1)) for d in os.listdir(MODELS) if (m := re.fullmatch(r'v(\d+)', d))])
prev = versions[-1] if versions else None
BASE = f'{MODELS}/v{prev}/best.pt' if prev and os.path.exists(f'{MODELS}/v{prev}/best.pt') else None
if prev == 1 and BASE is None and os.path.exists(f'{MODELS}/v1/hoverlogix_v1.pt'):
    BASE = f'{MODELS}/v1/hoverlogix_v1.pt'
BASE = BASE or 'yolo26n.pt'
NEW = (prev or 1) + 1
print('starting from', BASE, '-> training version', f'v{NEW}')
print('exports:', sorted(os.listdir(EXPORTS)))

In [ ]:
# 3. Merge all exports (a newer export of the same photo replaces the older one)
import zipfile, shutil, json
from pathlib import Path
RAW, DS = Path('/content/raw'), Path('/content/dataset')
shutil.rmtree(RAW, ignore_errors=True); shutil.rmtree(DS, ignore_errors=True)
(DS / 'all/images').mkdir(parents=True); (DS / 'all/labels').mkdir(parents=True)
photos, attrs, classes = {}, {}, None
for z in sorted(glob.glob(f'{EXPORTS}/*.zip')):
    out = RAW / Path(z).stem
    zipfile.ZipFile(z).extractall(out)
    for root in out.iterdir():
        if (root / 'classes.txt').exists():
            classes = [c.strip() for c in (root / 'classes.txt').read_text().splitlines() if c.strip()]
        for img in (root / 'images').glob('*.jpg'):
            pid = img.stem.rsplit('_', 1)[-1]
            photos[pid] = (img, root / 'labels' / (img.stem + '.txt'))
        if (root / 'attributes.jsonl').exists():
            for line in (root / 'attributes.jsonl').read_text().splitlines():
                if line.strip():
                    a = json.loads(line); attrs[a['id']] = a
for pid, (img, lab) in photos.items():
    shutil.copy(img, DS / 'all/images' / img.name)
    shutil.copy(lab, DS / 'all/labels' / lab.name)
print(len(photos), 'reviewed photos;', 'classes:', classes)
counts = [0] * len(classes)
for lab in (DS / 'all/labels').glob('*.txt'):
    for line in lab.read_text().splitlines():
        if line.strip(): counts[int(line.split()[0])] += 1
for c, n in zip(classes, counts): print(f'  {c:16s} {n}')
lean = sum(1 for a in attrs.values() for p in a.get('pallets', []) if p.get('condition') in ('leaning', 'overhang'))
print('pallets marked leaning/overhang:', lean)

In [ ]:
# 4. Train / validation split (by photo, stratified by source)
import random
random.seed(SEED)
by_source = {}
for img in sorted((DS / 'all/images').glob('*.jpg')):
    pid = img.stem.rsplit('_', 1)[-1]
    by_source.setdefault(attrs.get(pid, {}).get('source', 'unknown'), []).append(img)
train, val = [], []
for src, imgs in by_source.items():
    random.shuffle(imgs)
    k = max(1, round(len(imgs) * VAL_SHARE)) if len(imgs) > 4 else 0
    val += imgs[:k]; train += imgs[k:]
(DS / 'train.txt').write_text('\n'.join(str(p) for p in train) + '\n')
(DS / 'val.txt').write_text('\n'.join(str(p) for p in (val or train[:5])) + '\n')
yaml = f"path: {DS}\ntrain: train.txt\nval: val.txt\nnames:\n" + ''.join(f'  {i}: {c}\n' for i, c in enumerate(classes))
(DS / 'data.yaml').write_text(yaml)
print(len(train), 'train /', len(val), 'val photos')

In [ ]:
# 5. Train
from ultralytics import YOLO
model = YOLO(BASE)
model.train(data=str(DS / 'data.yaml'), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=-1, device=0,
            project='/content/runs', name=f'v{NEW}', exist_ok=True,
            degrees=3, scale=0.4, mosaic=1.0, close_mosaic=10, fliplr=0.5, seed=SEED)

In [ ]:
# 6. Evaluate on the validation photos
best = YOLO(f'/content/runs/v{NEW}/weights/best.pt')
m = best.val(data=str(DS / 'data.yaml'), imgsz=IMGSZ, device=0, split='val')
per_class = {classes[i]: {'mAP50': round(float(m.box.ap50[j]), 3), 'mAP50_95': round(float(m.box.ap[j]), 3)}
             for j, i in enumerate(m.box.ap_class_index)}
metrics = {'version': f'v{NEW}', 'base': BASE, 'photos': len(photos), 'train': len(train), 'val': len(val),
           'mAP50': round(float(m.box.map50), 3), 'mAP50_95': round(float(m.box.map), 3), 'per_class': per_class,
           'classes': classes, 'imgsz': IMGSZ}
print(json.dumps(metrics, indent=1))

In [ ]:
# 7. Export (ONNX for the review page and the backend) and save everything to Drive
onnx_path = best.export(format='onnx', imgsz=IMGSZ, opset=17, simplify=True)
out = Path(MODELS) / f'v{NEW}'
out.mkdir(parents=True, exist_ok=True)
shutil.copy(f'/content/runs/v{NEW}/weights/best.pt', out / 'best.pt')
shutil.copy(onnx_path, out / f'hoverlogix_v{NEW}_{IMGSZ}.onnx')
(out / 'metrics.json').write_text(json.dumps(metrics, indent=1))
for f in ['results.csv', 'confusion_matrix.png', 'results.png']:
    p = Path(f'/content/runs/v{NEW}') / f
    if p.exists(): shutil.copy(p, out / f)
bundle = Path(MODELS) / f'hoverlogix_model_v{NEW}.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in out.iterdir(): z.write(f, f'v{NEW}/{f.name}')
print('Saved to', out)
print('Attach this file in the chat with Claude:', bundle, f'({bundle.stat().st_size/1e6:.1f} MB)')

## Optional: Android model files
The Android app uses LiteRT (`.tflite`). Claude can make these from `best.pt`; or run the cell below
(it installs a large converter and takes ~10 minutes).

In [ ]:
# 8. (optional) LiteRT exports for the Android app
!pip -q install litert-torch ai-edge-litert
for size, name in [(640, 'hoverlogix_fast_640.tflite'), (960, 'hoverlogix_accurate_960.tflite')]:
    p = best.export(format='litert', imgsz=size)
    shutil.copy(p, out / name)
print(sorted(os.listdir(out)))